# Train a dedicated Indian drivable-area model
Targets: global drivable IoU ≥95%, pixel precision ≥98%, recall ≥95%. **These are targets, not achieved scores.** At 98% precision, 95% IoU requires about 96.88% recall.

Completed reference run: **92.26% drivable IoU, 97.47% pixel precision and 94.52% recall** on all 204 official validation images. This clean notebook reproduces a separately named experiment; identical results are not guaranteed.

Select **Runtime → Change runtime type → GPU**. Upload only your licensed `idd-lite.tar.gz`. Setup clones the public repository and uses its published frozen training split. Do not upload video or weights for this experiment.

The frozen split uses 1,213 official-training images for fitting, 190 for development, and keeps all 204 official validation images out of training/checkpoint selection. This is IDD Lite, not the IDD20k leaderboard. Lane markings and potholes are separate tasks.

A GPU run may be interrupted by Colab limits. No time estimate or target score is guaranteed. Download the results before ending the session; `/content` is temporary. The current trainer does not resume interrupted runs.

In [ ]:
import torch
assert torch.cuda.is_available(), 'A GPU runtime is required; CPU fallback is intentionally disabled.'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)
from google.colab import files
files.upload()  # Select your licensed IDD Lite archive only.


In [ ]:
import hashlib, tarfile, shutil, subprocess
from pathlib import Path
def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with tarfile.open(archive, 'r:gz') as handle:
        members = handle.getmembers()
        assert sum(member.size for member in members) < 512 * 1024**2, 'Unexpected archive expansion'
        for member in members:
            assert member.isfile() or member.isdir(), 'Links or special files not allowed'
            path = Path(member.name)
            assert not path.is_absolute() and '..' not in path.parts, 'Unsafe member path'
            (destination / path).resolve().relative_to(destination)
        assert not destination.exists(), 'Use a fresh extraction directory'
        destination.mkdir(parents=True)
        handle.extractall(destination, members=members, filter='data')
dataset = Path('/content/idd-lite.tar.gz')
assert hashlib.sha256(dataset.read_bytes()).hexdigest() == '064f92c46a1141f290dd2a52f4c84174381445baf9f6d6bf471be0aa006601a1', 'Dataset differs from frozen experiment'
repository = Path('/content/roadsense')
assert not repository.exists(), 'Use a fresh Colab runtime; do not overwrite an existing checkout'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Harsh-ryUk/RoadSense-India.git', str(repository)], check=True)
print('Source commit:', subprocess.check_output(['git', '-C', str(repository), 'rev-parse', 'HEAD'], text=True).strip())
frozen_split = repository / 'benchmarks/idd_lite_finetuned_20261004/training_splits.json'
assert hashlib.sha256(frozen_split.read_bytes()).hexdigest() == '1e30c0de0df837c0bb615b4b5ed7a42e5132e5c18a2733409bfe92b8e47433da', 'Frozen training split changed'
(repository / 'training').mkdir()
shutil.copyfile(frozen_split, repository / 'training/splits.json')
safe_extract(dataset, '/content/idd_data')
%cd /content/roadsense
%pip install -q -r requirements-training.txt
# If installation changes already-imported packages, restart the runtime before training.


## Reproduce the fixed training recipe
50 epochs, SegFormer-B0 with a new two-class head, 512×512 input, batch 8, AdamW at 6e-5, seed 42. Development metrics select the checkpoint; **they are raw model scores, not deployed/official-validation scores**. Do not alter the final configuration after viewing official-validation results. If memory is insufficient, record a separately named smaller-batch experiment.

In [ ]:
import subprocess
subprocess.run(['python', '-u', 'scripts/train_drivable_idd_lite.py', '--root', '/content/idd_data/idd20k_lite', '--manifest', 'training/splits.json', '--output', '/content/drivable_run', '--device', 'cuda:0', '--epochs', '50', '--batch', '8', '--input-size', '512', '--lr', '0.00006', '--seed', '42'], check=True)


## Evaluate the selected checkpoint separately
Only run this after the fixed experiment finishes. This includes the deployed mask postprocessor, all 204 images and all failed predictions. The official validation set already has a published baseline; it is not a previously unseen independent test set. Repeated changes guided by this score require a new untouched test set for a generalization claim.

In [ ]:
subprocess.run(['python', 'scripts/prepare_idd_lite_eval.py', '--root', '/content/idd_data/idd20k_lite', '--output', '/content/drivable_run/official_val_manifest.json'], check=True)
subprocess.run(['python', '-u', 'scripts/evaluate_accuracy.py', '--manifest', '/content/drivable_run/official_val_manifest.json', '--config', '/content/drivable_run/evaluation_config.yaml', '--device', 'cuda:0', '--threads', '2', '--output', '/content/drivable_run/official_val_result.json'], check=True)
subprocess.run(['python', 'scripts/audit_accuracy_report.py', '--report', '/content/drivable_run/official_val_result.json', '--manifest', '/content/drivable_run/official_val_manifest.json'], check=True)
import json
result = json.loads(Path('/content/drivable_run/official_val_result.json').read_text())
summary = json.loads(Path('/content/drivable_run/training_summary.json').read_text())
assert result['models']['segformer_local_files_sha256']['model.safetensors'] == summary['best_weights_sha256'], 'Evaluated checkpoint differs from selected training checkpoint'
metrics = result['binary_road']
print('Measured deployed official-val metrics:', metrics)
met = {key: metrics[key] is not None and metrics[key] >= threshold for key, threshold in [('IoU', 0.95), ('precision', 0.98), ('recall', 0.95)]}
print('Targets met:', met, 'All met:', all(met.values()))


In [ ]:
# Private export includes selected weights and run metadata, not dataset images/masks.
import shutil
archive = shutil.make_archive('/content/drivable_run_results', 'gztar', '/content', 'drivable_run')
files.download(archive)
